# 📐 Analysis After Normalization — 16 Stations
**Data:** `data/normalized/norm_station_{id}.csv`  
**Strategy:** Clip → log1p → Scale (per-station, fit on train split)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.grid':True,'grid.alpha':0.3})

SELECTED_STATIONS = [1,3,4,7,9,12,13,15,16,17,18,24,27,29,31,32]
NORM_DIR = 'data/normalized'
CLEAN_DIR = 'data/clean'

ALL_SCALED = [
    'pm25','pm10','co','o3','no2','so2',
    'temp','rh','dewpt','precip','clouds',
    'wind_spd','wind_gusts','wind_sin','wind_cos',
    'soil_temp_0_7','soil_moist_0_7',
    'oxidation_potential','pollution_load',
    'no2_so2_ratio','humid_sulfate_risk','dpd',
    'thermal_stability','stagnation_index','dust_source_potential',
]

POLLUTANTS = ['pm25','pm10','co','o3','no2','so2']
WEATHER    = ['temp','rh','dewpt','precip','clouds','wind_spd','wind_gusts']
ENGINEERED = ['oxidation_potential','pollution_load','no2_so2_ratio',
              'humid_sulfate_risk','dpd','thermal_stability',
              'stagnation_index','dust_source_potential']

norm_dfs  = []
norm_meta = []
for sid in SELECTED_STATIONS:
    df = pd.read_csv(f'{NORM_DIR}/norm_station_{sid}.csv', parse_dates=['timestamp'])
    df = df.set_index('timestamp').sort_index()
    norm_dfs.append(df)
    norm_meta.append({'station_id':sid,
                       'province':df['province'].iloc[0],
                       'district':df['district'].iloc[0]})

_sid2i = {m['station_id']:i for i,m in enumerate(norm_meta)}
def gdf(sid): return norm_dfs[_sid2i[sid]]

all_norm = pd.concat([df.reset_index() for df in norm_dfs], ignore_index=True)
print(f'Loaded {len(all_norm):,} rows | {len(all_norm.columns)} cols')
all_norm[ALL_SCALED].describe().round(3)

---
## 1. Phân phối sau chuẩn hoá — All Features (Histogram + KDE)

In [ ]:
PLOT_COLS = [c for c in ALL_SCALED if c in all_norm.columns and c not in ['wind_sin','wind_cos']]
n = len(PLOT_COLS)
ncols = 5
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(26, nrows * 3.5))
axes = axes.flatten()

for i, col in enumerate(PLOT_COLS):
    ax = axes[i]
    vals = all_norm[col].dropna()
    ax.hist(vals, bins=80, density=True, alpha=0.45,
            color=plt.cm.tab20(i/len(PLOT_COLS)), edgecolor='none')
    vals.plot.kde(ax=ax, color='black', lw=1.8)
    ax.axvline(0,  color='red',    lw=1, ls='--', alpha=0.6)
    ax.axvline(-2, color='orange', lw=0.8, ls=':', alpha=0.5)
    ax.axvline(2,  color='orange', lw=0.8, ls=':', alpha=0.5)
    ax.set_title(col, fontsize=9, fontweight='bold')
    sk = round(vals.skew(), 2)
    ku = round(vals.kurt(), 2)
    ax.text(0.97, 0.97, f'skew={sk}\nkurt={ku}',
            transform=ax.transAxes, ha='right', va='top', fontsize=7,
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.8))

for j in range(i+1, len(axes)): axes[j].set_visible(False)

fig.suptitle('Phân phối sau chuẩn hoá (đường đỏ=0, cam=±2σ)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('norm_dist_histogram.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Violin Plot Cross-Station — Pollutants (sau chuẩn hoá)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(24, 12))
axes = axes.flatten()
station_labels = [f"T{m['station_id']}\n{m['district'][:7]}" for m in norm_meta]

for i, col in enumerate(POLLUTANTS):
    ax = axes[i]
    data_list = []
    for df in norm_dfs:
        if col in df.columns:
            data_list.append(df[col].dropna().values)
        else:
            data_list.append(np.array([0]))

    vp = ax.violinplot(data_list, positions=range(len(norm_meta)),
                        showmedians=True, showextrema=False)
    colors = sns.color_palette('tab20', len(norm_meta))
    for body, color in zip(vp['bodies'], colors):
        body.set_facecolor(color); body.set_alpha(0.7)
    vp['cmedians'].set_color('black'); vp['cmedians'].set_linewidth(2)

    ax.axhline(0, color='red', lw=1, ls='--', alpha=0.7, label='mean=0')
    ax.set_xticks(range(len(station_labels)))
    ax.set_xticklabels(station_labels, rotation=40, ha='right', fontsize=7)
    ax.set_title(col.upper(), fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)

fig.suptitle('Violin Plot Pollutants Cross-Station (sau chuẩn hoá)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('norm_violin_pollutants.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Boxplot — 8 Engineered Features Cross-Station (sau chuẩn hoá)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(28, 12))
axes = axes.flatten()

for i, col in enumerate(ENGINEERED):
    ax = axes[i]
    data_list = [df[col].dropna().values for df in norm_dfs if col in df.columns]

    bp = ax.boxplot(data_list, patch_artist=True, notch=False,
                    medianprops=dict(color='black', lw=2),
                    flierprops=dict(marker='.', markersize=2, alpha=0.3))

    colors = sns.color_palette('tab20', len(norm_meta))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color); patch.set_alpha(0.7)

    ax.axhline(0,  color='red',    lw=1, ls='--', alpha=0.7)
    ax.axhline(2,  color='orange', lw=0.8, ls=':', alpha=0.5)
    ax.axhline(-2, color='orange', lw=0.8, ls=':', alpha=0.5)
    ax.set_xticks(range(1, len(station_labels)+1))
    ax.set_xticklabels(station_labels, rotation=40, ha='right', fontsize=7)
    ax.set_title(col, fontsize=9, fontweight='bold')

fig.suptitle('Boxplot Engineered Features Cross-Station (đỏ=0, cam=±2σ)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('norm_boxplot_engineered.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Before vs After Normalization — KDE (Trạm 1 & 7 minh hoạ)

In [ ]:
DEMO_SID = [1, 7]   # HN Cau Giay, HCM Q1
DEMO_COLS = ['pm25', 'co', 'no2_so2_ratio', 'oxidation_potential',
             'temp', 'precip', 'stagnation_index', 'dust_source_potential']

fig, axes = plt.subplots(len(DEMO_COLS), len(DEMO_SID), figsize=(16, len(DEMO_COLS)*3))

for col_idx, sid in enumerate(DEMO_SID):
    df_clean = pd.read_csv(f'{CLEAN_DIR}/clean_station_{sid}.csv', parse_dates=['timestamp']).set_index('timestamp')
    df_norm_s = gdf(sid)
    meta = norm_meta[_sid2i[sid]]

    for row_idx, col in enumerate(DEMO_COLS):
        ax = axes[row_idx][col_idx]
        if col not in df_clean.columns or col not in df_norm_s.columns:
            ax.set_visible(False); continue

        raw_vals  = df_clean[col].dropna()
        norm_vals = df_norm_s[col].dropna()

        # Clip raw để KDE không bị kéo quá
        q1r, q99r = raw_vals.quantile(0.01), raw_vals.quantile(0.99)
        raw_vals.clip(q1r, q99r).plot.kde(
            ax=ax, color='firebrick', lw=2, ls='--', label='Before')
        norm_vals.plot.kde(
            ax=ax, color='steelblue', lw=2.5, label='After')

        ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)
        if col_idx == 0:
            ax.set_ylabel(col, fontsize=8)
        if row_idx == 0:
            ax.set_title(f"T{sid} {meta['district']}", fontsize=10, fontweight='bold')
        ax.legend(fontsize=7)
        ax.tick_params(labelsize=7)

fig.suptitle('Before vs After Normalization (KDE) — Trạm 1 & 7',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('norm_before_after_kde.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Correlation Heatmap (Spearman) — Sau chuẩn hoá (16 Stations)

In [ ]:
CORR_COLS = [c for c in ALL_SCALED if c in norm_dfs[0].columns and c != 'wind_sin']

fig, axes = plt.subplots(4, 4, figsize=(36, 32))
axes = axes.flatten()

for i, (df, meta) in enumerate(zip(norm_dfs, norm_meta)):
    ax = axes[i]
    cols = [c for c in CORR_COLS if c in df.columns]
    corr = df[cols].corr(method='spearman')
    sns.heatmap(corr, ax=ax, annot=True, fmt='.1f',
                cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                linewidths=0.3, annot_kws={'size':5}, cbar=False)
    ax.set_title(f"T{meta['station_id']} {meta['district']}\n({meta['province']})",
                 fontsize=9, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=6)
    ax.tick_params(axis='y', rotation=0,  labelsize=6)

fig.suptitle('Spearman Correlation (Sau Chuẩn hoá) — 16 Stations',
             fontsize=18, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig('norm_corr_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. QQ Plots — Kiểm tra phân phối sau chuẩn hoá có gần Gaussian không

In [ ]:
from scipy import stats

CHECK_COLS = ['pm25','co','no2','precip','oxidation_potential',
              'pollution_load','no2_so2_ratio','stagnation_index']

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()
df_ref = norm_dfs[0]   # Trạm 1 - HN Cau Giay

for i, col in enumerate(CHECK_COLS):
    ax = axes[i]
    if col not in df_ref.columns:
        ax.set_visible(False); continue

    vals = df_ref[col].dropna().values
    sample = np.random.choice(vals, min(3000, len(vals)), replace=False)
    (osm, osr), (slope, intercept, r) = stats.probplot(sample, dist='norm')
    ax.scatter(osm, osr, alpha=0.3, s=5, color='steelblue')
    ax.plot(osm, slope * np.array(osm) + intercept, color='red', lw=2)
    ax.set_title(f'{col}\nr²={r**2:.3f}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Theoretical Quantiles', fontsize=8)
    ax.set_ylabel('Sample Quantiles', fontsize=8)
    # Thêm Shapiro-Wilk trên subset nhỏ
    if len(sample) >= 20:
        sub = sample[:500] if len(sample) > 500 else sample
        _, p = stats.shapiro(sub)
        ax.text(0.05, 0.95, f'Shapiro p={p:.3f}',
                transform=ax.transAxes, fontsize=7,
                color='green' if p > 0.05 else 'red',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.8))

fig.suptitle('QQ Plots (Gaussian) — Trạm 1 HN (r² gần 1 = gần chuẩn)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('norm_qq_plots.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Train / Val / Test — Kiểm tra phân phối PM2.5 không bị shift

In [ ]:
SPLIT_STATIONS = [1, 7, 24, 18]   # HN, HCM, Bien Hoa, Can Tho
SPLIT_COLORS   = {'train':'steelblue','val':'orange','test':'firebrick'}

fig, axes = plt.subplots(1, len(SPLIT_STATIONS), figsize=(22, 5))

for ax, sid in zip(axes, SPLIT_STATIONS):
    df = gdf(sid)
    meta = norm_meta[_sid2i[sid]]
    for split, color in SPLIT_COLORS.items():
        vals = df[df['split'] == split]['pm25'].dropna()
        if len(vals) > 10:
            vals.plot.kde(ax=ax, color=color, lw=2,
                          label=f'{split} (n={len(vals):,})')
    ax.set_title(f'T{sid} {meta["district"]}', fontsize=10, fontweight='bold')
    ax.set_xlabel('PM2.5 (scaled)'); ax.legend(fontsize=8)
    ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)

fig.suptitle('PM2.5 Distribution: Train / Val / Test (sau chuẩn hoá)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('norm_split_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

# Summary table
print('\nPM2.5 stats per split (Tram 1 - HN):')
df1 = gdf(1)
print(df1.groupby('split')['pm25'].describe().round(3))

---
## 8. 🔬 Deep-Dive: `no2_so2_ratio` — Root Cause & Fix

**Vấn đề:** skew = 250, max = 1.54×10⁸ → clip + RobustScaler không đủ.

**Root cause:** SO₂ = 0 tại vùng rural/ven biển → NO2/(SO2+ε) → ∞

**Giải pháp:** Thay ratio bằng `log(NO2+1) − log(SO2+1)` (log-difference)

In [ ]:
# ── 8.1 Stats so2 near-zero & ratio anatomy ──────────────────────────────────
all_clean = pd.concat([
    pd.read_csv(f'data/clean/clean_station_{sid}.csv',
                usecols=['province','no2','so2','no2_so2_ratio']).assign(station_id=sid)
    for sid in [1,3,4,7,9,12,13,15,16,17,18,24,27,29,31,32]
], ignore_index=True)

ratio = all_clean['no2_so2_ratio'].dropna()
print('=== no2_so2_ratio stats (RAW) ===')
for p in [50, 90, 95, 99, 99.5]:
    print(f'  p{p:<5} = {ratio.quantile(p/100):>18,.2f}')
print(f'  max    = {ratio.max():>18,.2f}')
print(f'  skew   = {ratio.skew():>18.2f}')

# SO2 near-zero
print('\n=== SO2 = 0 per province (%)')
print(all_clean.groupby('province').apply(
    lambda g: round((g['so2']==0).sum()/len(g.dropna(subset=['so2']))*100, 2)
).sort_values(ascending=False).to_string())

# Figure: 3 panels
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Panel 1: log1p(raw)
ax = axes[0]
np.log1p(ratio.clip(lower=0)).plot.hist(
    ax=ax, bins=100, color='firebrick', alpha=0.7, density=True)
ax.set_title(f'log1p(ratio) RAW\nskew={np.log1p(ratio.clip(lower=0)).skew():.2f}',
             fontweight='bold')
ax.set_xlabel('log1p(ratio)')

# Panel 2: clip99 + RobustScaler (hiện tại)
ax = axes[1]
from sklearn.preprocessing import RobustScaler
q99 = ratio.quantile(0.99)
clipped = ratio.clip(upper=q99)
sc = RobustScaler(); scaled = sc.fit_transform(clipped.values.reshape(-1,1)).flatten()
pd.Series(scaled).plot.hist(ax=ax, bins=100, color='orange', alpha=0.7, density=True)
ax.set_title(f'Clip99 + RobustScaler (CURRENT)\nskew={pd.Series(scaled).skew():.2f}',
             fontweight='bold')

# Panel 3: log-difference
ax = axes[2]
log_diff = (np.log1p(all_clean['no2'].clip(lower=0)) -
            np.log1p(all_clean['so2'].clip(lower=0))).dropna()
from sklearn.preprocessing import StandardScaler
ld_scaled = StandardScaler().fit_transform(log_diff.values.reshape(-1,1)).flatten()
pd.Series(ld_scaled).plot.hist(ax=ax, bins=100, color='steelblue', alpha=0.7, density=True)
ax.set_title(f'log(NO2+1) − log(SO2+1) + Standard  [ĐỀ XUẤT]\nskew={pd.Series(ld_scaled).skew():.2f}',
             fontweight='bold', color='steelblue')

fig.suptitle('no2_so2_ratio: Anatomy & Alternative', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ratio_anatomy.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.2 QQ Plot so sánh 3 cách biểu diễn ────────────────────────────────────
from scipy import stats as scipy_stats
from sklearn.preprocessing import RobustScaler, StandardScaler

df1_clean = pd.read_csv('data/clean/clean_station_1.csv',
                         parse_dates=['timestamp']).set_index('timestamp')

raw_ratio = df1_clean['no2_so2_ratio'].dropna()
q99 = raw_ratio.quantile(0.99)
m1 = RobustScaler().fit_transform(raw_ratio.clip(upper=q99).values.reshape(-1,1)).flatten()

log_diff_raw = (np.log1p(df1_clean['no2'].clip(lower=0)) -
                np.log1p(df1_clean['so2'].clip(lower=0))).dropna()
m3 = StandardScaler().fit_transform(log_diff_raw.values.reshape(-1,1)).flatten()

METHODS = [
    ('Clip99 + RobustScaler (HIỆN TẠI)', m1, 'firebrick'),
    ('log(NO2+1) − log(SO2+1) + Standard (ĐỀ XUẤT)', m3, 'steelblue'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col_idx, (label, series, color) in enumerate(METHODS):
    sample = np.random.choice(series, min(3000, len(series)), replace=False)

    # QQ Plot
    ax = axes[0][col_idx]
    (osm, osr), (slope, intercept, r) = scipy_stats.probplot(sample, dist='norm')
    ax.scatter(osm, osr, alpha=0.3, s=6, color=color)
    ax.plot(osm, slope * np.array(osm) + intercept, 'k-', lw=2)
    ax.set_title(f'{label}\nr²={r**2:.4f}', fontsize=10, fontweight='bold', color=color)

    # Histogram
    ax2 = axes[1][col_idx]
    ax2.hist(sample, bins=80, density=True, alpha=0.55, color=color, edgecolor='none')
    pd.Series(sample).plot.kde(ax=ax2, color=color, lw=2)
    ax2.axvline(0, color='red', lw=1, ls='--', alpha=0.6)
    sk = pd.Series(sample).skew()
    ax2.text(0.97,0.97,f'skew={sk:.3f}', transform=ax2.transAxes,
             ha='right', va='top', fontsize=9, bbox=dict(fc='white', alpha=0.8))

fig.suptitle('no2_so2_ratio: QQ + Distribution so sánh 2 cách (Trạm 1 HN)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ratio_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nR² Summary:')
for label, series, _ in METHODS:
    s = np.random.choice(series, min(3000,len(series)), replace=False)
    _, _, r = scipy_stats.probplot(s, dist='norm')[1]
    print(f'  [{r**2:.4f}] {label}')

In [ ]:
# ── 8.3 Time series SO2 spikes tại trạm rural ────────────────────────────────
FOCUS = [13, 27, 29]   # Da Nang, Nam Dinh, Nghe An
fig, axes = plt.subplots(len(FOCUS), 1, figsize=(22, 4*len(FOCUS)))

for ax, sid in zip(axes, FOCUS):
    df_c = pd.read_csv(f'data/clean/clean_station_{sid}.csv',
                        parse_dates=['timestamp']).set_index('timestamp')
    meta = next(m for m in norm_meta if m['station_id'] == sid)
    daily = df_c[['no2','so2','no2_so2_ratio']].resample('D').median()

    q95 = daily['no2_so2_ratio'].quantile(0.95)
    clipped = daily['no2_so2_ratio'].clip(upper=q95)

    ax2 = ax.twinx()
    ax.fill_between(daily.index, daily['no2'].fillna(0), alpha=0.45,
                    color='royalblue', label='NO2')
    ax.fill_between(daily.index, daily['so2'].fillna(0), alpha=0.45,
                    color='tomato', label='SO2')
    ax2.plot(clipped.index, clipped, color='black', lw=1, label='ratio (clip p95)')

    spikes = daily['no2_so2_ratio'] > daily['no2_so2_ratio'].quantile(0.99)
    ax2.scatter(daily.index[spikes], clipped[spikes],
                color='red', s=20, zorder=5, label=f'spike>p99 ({spikes.sum()})')

    ax.set_title(f'T{sid} {meta["district"]} ({meta["province"]}) '
                 f'— SO2 thấp → ratio spike',
                 fontweight='bold', fontsize=10)
    ax.set_ylabel('Concentration (µg/m³)')
    ax2.set_ylabel('Ratio (clipped)')
    h1,l1 = ax.get_legend_handles_labels()
    h2,l2 = ax2.get_legend_handles_labels()
    ax.legend(h1+h2, l1+l2, fontsize=8, loc='upper right')

fig.suptitle('Time Series: NO2, SO2 & Ratio — Rural/Coastal',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ratio_timeseries.png', dpi=120, bbox_inches='tight')
plt.show()

### ✅ Kết luận: Thay `no2_so2_ratio` → `no2_so2_log_diff`

| | Cách hiện tại | Đề xuất |
|---|---|---|
| **Công thức** | `NO2 / (SO2 + 1e-6)` | `log(NO2+1) − log(SO2+1)` |
| **Skewness** | ~250 | ~0.3–0.5 |
| **r² QQ** | ~0.5 | ~0.95 |
| **SO2=0** | → ∞ | → log(NO2+1) ✅ |

**→ Cần cập nhật `preprocessing.py` → `create_weather_features()` → rename feature → chạy lại pipeline.**